In [1]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:
import sys
sys.path.append("/content/drive/MyDrive/stocks/duet_class")

from pipeline.config import DUETConfig
from pipeline import preprocess, train, evaluate, predict
from duet.model import DUETModel
from torch.utils.data import DataLoader, TensorDataset
import torch
import joblib
import random

import numpy as np
from sklearn.utils.class_weight import compute_class_weight

In [2]:
# Настройки
joblib_path = "/content/drive/MyDrive/stocks/duet_class/data/DOGE_1m_small_class.joblib"
device = "cuda" if torch.cuda.is_available() else "cpu"

config = DUETConfig(
    # =========================
    # Общие параметры
    # =========================
    timestamp_col = "timestamp",
    features = [
       'Open', 'High', 'Low', 'Close', #'Volume',
      #  'sigma_log_return_7', 'sigma_bollinger_7', 'sigma_true_range_7', 'min_extr_distance_7',
      #  'max_extr_distance_7', 'efficiency_ratio_7',
       'sigma_log_return_14', 'sigma_bollinger_14', 'sigma_true_range_14', 'min_extr_distance_14',
       'max_extr_distance_14', 'efficiency_ratio_14'
    ],                           # Название колонок для обучения
    forecast = 'pivots',         # Название колонки с таргетом
    not_to_normalise = [],       # Название колонок, которые НЕ НАДО нормализовать
    scaler = 'MINMAX',         # Тип нормализации (STD, MINMAX, QUANT)
    predict_type = 'next',     # Детекция "detect" или предикт "next" следующей свечи
    seq_len = 32,                # Длина входной последовательности
    num_classes = 3,             # кол-во классов
    patch_len = 8,               # Длина патча (TCM)
    stride = 4,                  # Шаг между патчами (TCM)
    moving_avg = 5,              # Размер окна скользящего сглаживания

    # =========================
    # Параметры модели
    # =========================
    d_model = 64,               # Размерность скрытого пространства в attention
    d_ff = 512,                 # Размерность feedforward слоя
    n_heads = 4,                # Количество голов в multi-head attention
    e_layers = 3,               # Количество слоев в encoder (CCM)
    dropout = 0.2,            # Dropout во всех слоях attention
    fc_dropout = 0.2,         # Dropout в выходном head слое
    activation = "relu",        # Активационная функция (relu, gelu, elu)
    num_experts = 4,            # Число экспертов (в Router, если используется)
    report_freq = 10,            # Частота появления confusion matrix

    # =========================
    # Режимы обработки
    # =========================
    CI = False,                 # Channel-Independent режим (если False — shared weights)
    use_router = True,          # Включить распределительный роутер
    timeenc = 1,                # Использовать time encoding (0 = без, 1 = sin/cos и т.п.)

    # =========================
    # Настройки обучения
    # =========================
    batch_size=32,          # можно немного увеличить при хорошем GPU
    epochs=20,             # больше эпох для более тонкой настройки
    learning_rate=1e-5,     # стандартный LR для Adam
    weight_decay=1e-5,
    patience=100,            # early stopping не слишком строгий

    # =========================
    # Прочее
    # =========================
    checkpoint_best = '/content/drive/MyDrive/stocks/duet_class/weights/best_val_acc_weights_001.pt',       # Путь для сохранения лучших по val_accuracy весов
    checkpoint_final = '/content/drive/MyDrive/stocks/duet_class/weights/final_weights_001.pt',      # Путь для сохранения финальных весов
    seed = 11,                  # Фиксированное зерно генератора случайных чисел
    verbose = True,            # Печать хода обучения
)

"""
Устанавливает seed для numpy, random, torch (вкл. CUDA).
Гарантирует воспроизводимость.
"""

random.seed(config.seed)
np.random.seed(config.seed)
torch.manual_seed(config.seed)
torch.cuda.manual_seed_all(config.seed)

In [3]:
# --- 1. Предобработка данных ---
# Загрузка данных
df = joblib.load(joblib_path)
df = preprocess.prepare_time_series(df, config) # проверка пропусков, установка datetime индекса
preprocess.check_data(df, config) # проверка на nan & inf

# --- 6. Инициализация модели ---
model = DUETModel(config).to(device)

# Загрузка весов

# model.load_state_dict(torch.load(config.checkpoint_best)) # загрузка лучших весов
model.load_state_dict(torch.load(config.checkpoint_final)) # загрузка финальных весов
model.eval()


Индекс уже в формате datetime
Инферированная частота: None
Пропусков в индексе не обнаружено
Окончательно удалено 0 строк содеражащих NaN


DUETModel(
  (tcm): TCM(
    (decomp): SeriesDecomposition(
      (avg_pool): AvgPool1d(kernel_size=(5,), stride=(1,), padding=(2,))
    )
    (extractor): LinearPatternExtractor(
      (projection): ModuleList()
      (dropout): Dropout(p=0.2, inplace=False)
      (shared_proj): Linear(in_features=16, out_features=64, bias=True)
    )
  )
  (ccm): CCM(
    (enc_layers): ModuleList(
      (0-2): 3 x EncoderLayer(
        (attn): MultiHeadSelfAttention(
          (qkv_proj): Linear(in_features=64, out_features=192, bias=True)
          (out_proj): Linear(in_features=64, out_features=64, bias=True)
          (dropout): Dropout(p=0.2, inplace=False)
        )
        (dropout1): Dropout(p=0.2, inplace=False)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (activation): ReLU()
        (ffn): Sequential(
          (0): Linear(in_features=64, out_features=512, bias=True)
          (1): ReLU()
          (2): Dropout(p=0.2, inplace=False)
          (3): Linear(in_

In [4]:
# Предикт одного окна
from pipeline.predict import predict_window

#текущая точка t (sequence берется влево от t)
t = 1000

x_window = df.iloc[t - config.seq_len + 1 : t + 1]
pred = predict_window(model, x_window, config, device=device)

print(pred)

[[0.24496095 0.1892486  0.5657904 ]]


In [5]:
from pipeline.predict import predict_dataset_batched

full_df = df.iloc[52:150]

# Предсказания
y_pred, y_probs = predict_dataset_batched(model, full_df, config, device="cuda")

print(f"Prediction labels array shape {y_pred.shape}")
print(f"Prediction probs array shape {y_probs.shape}")


Prediction labels array shape (98,)
Prediction probs array shape (98, 3)
